<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Advance_Deep_Learning_Final_Exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advance Deep Learning Final Exam

Fill in the blanks (`__________`) to complete each coding task.  
Each question carries **5 marks** (20 questions = **100 marks**).

email: alfattaulislamss@gmail.com

In [ ]:
# Run this cell first
!pip install -q gensim
import warnings; warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 55.9 MB/s eta 0:00:00


---
# Part 1: Simple RNN

## Q1: RNN Single Time-Step (Forward Propagation)  *(5 marks)*
 Complete the forward step.

In [ ]:
import numpy as np

x_t    = np.array([1.0, 2.0])
h_prev = np.array([0.0, 0.0])
W_xh = np.array([[0.5, 0.1], [0.2, 0.3]])
W_hh = np.array([[0.1, 0.0], [0.0, 0.1]])
b    = np.array([0.0, 0.0])

h_t = np.tanh(np.dot(W_xh, x_t) + np.dot(W_hh, h_prev) + b)
print(h_t)

[0.60436778 0.66403677]


## Q2: Unrolling an RNN over a Sequence  *(5 marks)*

An RNN reuses the **same weights** at every time step and passes the hidden state forward. Fill in the loop.

In [ ]:
import numpy as np

inputs = [np.array([1.0]), np.array([0.5]), np.array([-1.0])]
W_xh, W_hh = 0.8, 0.5
h = np.zeros(1)

for x in inputs:
    h = np.tanh(W_xh * x + W_hh * h)
    print("h =", h)

h = [0.66403677]
h = [0.62429862]
h = [-0.45250889]


## Q3: nn.RNN Layer in PyTorch  *(5 marks)*

Build an `nn.RNN` with **3 input features** and **4 hidden units** (batch first). Output of all steps should be `[2, 5, 4]`.

Parameter formula (PyTorch has 2 bias vectors): `hidden × (input + hidden + 2)`.

In [ ]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size= 3, hidden_size= 4, batch_first=True)

x = torch.ones(2, 5, 3)            # batch=2, 5 time steps, 3 features
output, h_n = rnn(x)

last_step = output[:,  -1, :]    # hidden state of the LAST time step

print("output shape   :", tuple(output.shape))
print("h_n shape      :", tuple(h_n.shape))
print("last step shape:", tuple(last_step.shape))
print("Parameters     :", sum(p.numel() for p in rnn.parameters()))

output shape   : (2, 5, 4)
h_n shape      : (1, 2, 4)
last step shape: (2, 4)
Parameters     : 36


## Q4: Backpropagation: tanh Derivative  *(5 marks)*

 Complete the code for the gradient through tanh  In BPTT

In [ ]:
import numpy as np

h  = np.array([0.5, -0.2, 0.9])      # hidden state after tanh
dh = np.array([1.0, 2.0, 0.5])       # gradient coming from the next layer

dh_raw = (1 - h ** 2) * dh
print(dh_raw)

[0.75  1.92  0.095]


## Q5: Backpropagation: Gradients of W_hh, b and h_prev (one step)  *(5 marks)*

Given `dh_raw`, compute the gradients: `dW_hh = dh_raw ⊗ h_prev`, `db = dh_raw`, `dh_prev = W_hhᵀ · dh_raw`.

In [ ]:
import numpy as np

W_hh   = np.array([[0.1, 0.2], [0.3, 0.4]])
h_prev = np.array([1.0, 2.0])
dh_raw = np.array([0.5, -1.0])

dW_hh   = np.outer(dh_raw, h_prev)
db      = dh_raw
dh_prev = W_hh.T @ dh_raw

print("dW_hh:\n", dW_hh)
print("db:", db)
print("dh_prev:", dh_prev)

dW_hh:
 [[ 0.5  1. ]
 [-1.  -2. ]]
db: [ 0.5 -1. ]
dh_prev: [-0.25 -0.3 ]


---
# Part 2: LSTM

## Q6: Forget Gate  *(5 marks)*


In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x_t    = np.array([1.0, 0.5])
h_prev = np.array([0.2, -0.1])
W_f = np.array([[0.5, 0.1, 0.3, -0.2],
                [0.2, -0.4, 0.6, 0.1]])
b_f = np.zeros(2)

concat = np.concat([h_prev, x_t])
f_t = sigmoid(np.dot(W_f, concat) + b_f)
print("concat:", concat)
print("f_t   :", np.round(f_t, 4))

concat: [ 0.2 -0.1  1.   0.5]
f_t   : [0.572  0.6748]


## Q7: Cell State Update  *(5 marks)*


In [ ]:
import numpy as np

f_t    = np.array([0.9, 0.1])
i_t    = np.array([0.5, 0.8])
g_t    = np.array([0.4, -0.6])     # candidate memory
c_prev = np.array([2.0, 3.0])

c_t = f_t * c_prev + i_t * g_t
print(c_t)

[ 2.   -0.18]


## Q8: Output Gate and Hidden State  *(5 marks)*



In [ ]:
import numpy as np

o_t = np.array([0.7, 0.2])
c_t = np.array([1.0, -2.0])

h_t = o_t * np.tanh(c_t)
print(np.round(h_t, 4))

[ 0.5331 -0.1928]


## Q9: LSTM Parameters (4 gates)  *(5 marks)*

An LSTM has **4 gates/blocks** (forget, input, candidate, output). PyTorch formula: `4 × (hidden × (hidden + input) + 2 × hidden)`.

Build an LSTM with hidden size 8, then verify with the manual formula.

In [ ]:
import torch
import torch.nn as nn

units, features = 8, 5
lstm = nn.LSTM(input_size=features, hidden_size=units, batch_first=True)

x = torch.ones(3, 10, features)     # batch=3, 10 steps, 5 features
output, (h_n, c_n) = lstm(x)

manual = 4 * (units * (units + features) + 2 * units)

print("output shape  :", tuple(output.shape))
print("h_n shape     :", tuple(h_n.shape))
print("c_n shape     :", tuple(c_n.shape))
print("PyTorch params:", sum(p.numel() for p in lstm.parameters()))
print("Manual formula:", manual)

output shape  : (3, 10, 8)
h_n shape     : (1, 3, 8)
c_n shape     : (1, 3, 8)
PyTorch params: 480
Manual formula: 480


## Q10: LSTM Mini Project: Sentiment Classifier  *(5 marks)*

Build a tiny sentiment classifier in PyTorch: **Embedding → LSTM(16) → Linear(16, 1) → sigmoid**. Complete the model and the training loop (only shapes are printed, so your output must match exactly).

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)

texts = ["i love this movie", "great film", "what a wonderful story",
         "i hate this movie", "terrible film", "what a boring story"]
labels = [1, 1, 1, 0, 0, 0]

vocab = sorted(set(" ".join(texts).split()))
word_index = {w: i + 1 for i, w in enumerate(vocab)}      # 0 is for padding

def encode(t, maxlen=5):
    ids = [word_index[w] for w in t.split()]
    return [0] * (maxlen - len(ids)) + ids               # pre-padding

X = torch.tensor([encode(t) for t in texts])               # shape (6, 5)
y = torch.tensor(labels, dtype=torch.float32)

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 8, padding_idx=0)
        self.lstm = nn.LSTM(8, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(self.embedding(x))
        return torch.sigmoid(self.fc(h_n[-1]))

model = SentimentLSTM(len(vocab) + 1)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    optimizer.zero_grad()
    pred = model(X).squeeze(1)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()

print("Vocab size      :", len(vocab))
print("First sentence  :", X[0].tolist())
print("X shape         :", tuple(X.shape))
print("Prediction shape:", tuple(model(X).shape))

Vocab size      : 13
First sentence  : [0, 6, 7, 11, 8]
X shape         : (6, 5)
Prediction shape: (6, 1)


---
# Part 3: NLP

## Q11: Text Preprocessing Pipeline  *(5 marks)*

Lowercase the text, remove punctuation using regex (keep only word characters and spaces), then tokenize.

In [ ]:
import re

text = "Natural Language Processing (NLP) is FUN!!! It's used in 100+ apps."

text = text.lower()
text = re.sub(r"[^\w\s]", "", text)
tokens = text.split()
print(tokens)

['natural', 'language', 'processing', 'nlp', 'is', 'fun', 'its', 'used', 'in', '100', 'apps']


## Q12: N-grams (Bigrams)  *(5 marks)*

Create bigrams (pairs of consecutive words) using `zip`.

In [ ]:
tokens = ["i", "love", "natural", "language", "processing"]

bigrams = list(zip(tokens, tokens[1:]))
print(bigrams)

[('i', 'love'), ('love', 'natural'), ('natural', 'language'), ('language', 'processing')]


## Q13: TF-IDF Vocabulary and Matrix  *(5 marks)*

Use `TfidfVectorizer`: learn the vocabulary and transform the documents in one step.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["NLP is fun", "NLP is powerful", "deep learning is powerful"]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)

print(vectorizer.get_feature_names_out())
print("Shape:", X.shape)
print(np.round(X.toarray()[0], 2))

['deep' 'fun' 'is' 'learning' 'nlp' 'powerful']
Shape: (3, 6)
[0.   0.72 0.43 0.   0.55 0.  ]


## Q14: Cosine Similarity between Vectors  *(5 marks)*

`cos(a, b) = (a · b) / (||a|| × ||b||)`. Same direction = 1, opposite = -1, perpendicular = 0.

In [ ]:
import numpy as np

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([1, 2, 3])
print("same direction :", round(cosine(a, np.array([2, 4, 6])), 2))
print("opposite       :", round(cosine(a, np.array([-1, -2, -3])), 2))
print("perpendicular  :", round(cosine(np.array([1, 0]), np.array([0, 1])), 2))

same direction : 1.0
opposite       : -1.0
perpendicular  : 0.0


## Q15: Word2Vec (Skip-Gram)  *(5 marks)*

Train **Skip-Gram** Word2Vec (`sg=1`) with **10-dimensional** vectors, window 2 and keep all words.

In [ ]:
# !pip install -q gensim
from gensim.models import Word2Vec

sentences = [
    ["i", "love", "nlp"],
    ["nlp", "is", "awesome"],
    ["machine", "learning", "is", "fun"],
    ["i", "love", "machine", "learning"]
]

model = Word2Vec(sentences, vector_size=10, window=2,
                 min_count=1, sg=1, seed=42, workers=1)

print("Vocabulary size:", len(model.wv.key_to_index))
print("Vector shape   :", model.wv["nlp"].shape)

Vocabulary size: 8
Vector shape   : (10,)


---
# Part 4: CNN

## Q16: Convolution Output Size Formula  *(5 marks)*

n = input size, f = filter size, p = padding, s = stride.

In [ ]:
def conv_output_size(n, f, p, s):
    return (n - f + 2 * p) // s + 1

print(conv_output_size(28, 3, 0, 1))
print(conv_output_size(32, 5, 2, 1))
print(conv_output_size(7, 3, 0, 2))

26
32
3


## Q17: Padding  *(5 marks)*

Add a border of zeros (padding = 1) around the image.

In [ ]:
import numpy as np

img = np.array([[1, 2],
                [3, 4]])

padded = np.pad(img, pad_width=1, mode="constant", constant_values=0)
print(padded)
print("Shape:", padded.shape)

[[0 0 0 0]
 [0 1 2 0]
 [0 3 4 0]
 [0 0 0 0]]
Shape: (4, 4)


## Q18: Convolution with Stride = 2  *(5 marks)*

Slide a 3×3 all-ones filter over a 5×5 image with **stride 2**. Fill in the blanks.

In [ ]:
import numpy as np

img = np.arange(1, 26).reshape(5, 5)
kernel = np.ones((3, 3))

stride = 2
out_size = (5 - 3) // stride + 1
out = np.zeros((out_size, out_size))

for i in range(out_size):
    for j in range(out_size):
        r, c = i * stride, j * stride
        patch = img[r:r+3, c:c+3]
        out[i, j] = np.sum(patch * kernel)

print(img)
print(out)

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]
 [21 22 23 24 25]]
[[ 63.  81.]
 [153. 171.]]


## Q19: Max Pooling vs Average Pooling  *(5 marks)*

Apply 2×2 pooling (stride 2) on a 4×4 feature map. `reshape(2,2,2,2)` splits it into four 2×2 blocks.

In [ ]:
import numpy as np

fm = np.array([[1, 3, 2, 4],
               [5, 6, 1, 2],
               [7, 2, 9, 0],
               [3, 4, 1, 8]])

blocks = fm.reshape(2, 2, 2, 2)
max_pool = blocks.max(axis=(1, 3))
avg_pool = blocks.mean(axis=(1, 3))

print("Max pooling:\n", max_pool)
print("Avg pooling:\n", avg_pool)

Max pooling:
 [[6 4]
 [7 9]]
Avg pooling:
 [[3.75 2.25]
 [4.   4.5 ]]


## Q20: CNN Mini Project: Image Classifier Architecture  *(5 marks)*

Build a CNN for 28×28 grayscale images (10 classes): **Conv2d(8 filters, 3×3) → ReLU → MaxPool(2×2) → Flatten → Linear(?, 10)**.

PyTorch uses *channels-first* format: `(batch, channels, height, width)`. Hint: the Linear layer input = channels × height × width after pooling.

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(1352, 10)
)

x = torch.ones(1, 1, 28, 28)     # one grayscale 28x28 image
for layer in model:
    x = layer(x)
    print(layer.__class__.__name__, tuple(x.shape))

print("Total parameters:", sum(p.numel() for p in model.parameters()))

Conv2d (1, 8, 26, 26)
ReLU (1, 8, 26, 26)
MaxPool2d (1, 8, 13, 13)
Flatten (1, 1352)
Linear (1, 10)
Total parameters: 13610
